# Video-DPO: Temporal Consistency Alignment for AnimateDiff

This notebook runs the complete Video-DPO pipeline end-to-end:
1. **Setup**: Install dependencies and configure environment
2. **Data Generation**: Create preference pairs using the "Jitter Loop" technique
3. **Training**: DPO optimization of LoRA weights on motion modules
4. **Inference**: Generate comparison videos (base vs DPO-aligned)
5. **Evaluation**: Compute temporal consistency metrics

**Requirements**: GPU runtime (A100 recommended, T4 works but slower)

---
## 1. Environment Setup

In [1]:
# Check GPU availability
!nvidia-smi

Thu Dec 11 04:55:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone the repository (uncomment if running from scratch)
# !git clone https://github.com/YOUR_USERNAME/Video-DPO.git
# %cd Video-DPO

# Or upload the project files to Colab and set the path
import os
PROJECT_ROOT = "/content/Video-DPO"  # Adjust this path as needed

# Create project directory if it doesn't exist
os.makedirs(PROJECT_ROOT, exist_ok=True)
%cd {PROJECT_ROOT}

/content/Video-DPO


In [3]:
# Install dependencies
!pip uninstall numpy -y
!pip install "numpy<2.0.0"

!pip install -q torch>=2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers>=0.26.0 transformers accelerate peft safetensors
!pip install -q pyyaml tqdm einops imageio[ffmpeg] pillow
!pip install -q opencv-python>=4.8.0 scipy

print("Dependencies installed successfully!")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
Dependencies installed successfully!


In [3]:
# Verify installations
import torch
import diffusers
import accelerate
import peft

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Diffusers version: {diffusers.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"PEFT version: {peft.__version__}")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
VRAM: 15.8 GB
Diffusers version: 0.35.2
Accelerate version: 1.12.0
PEFT version: 0.18.0


---
## 2. Configuration

We'll use a Colab-optimized configuration with reduced data size for faster execution.

In [ ]:
# Configuration for optimal training
# Run this after initial testing shows the pipeline works

import os
import torch

# GPU Selection
GPU_TYPE = "A100"  # Options: "T4", "L4", "A100"

# Optimized settings for best quality
GPU_CONFIGS = {
    "T4": {
        "num_frames": 8,
        "resolution": 256,
        "num_pairs": 10,
        "max_train_steps": 50,
        "lora_rank": 4,
    },
    "L4": {
        "num_frames": 16,
        "resolution": 512,
        "num_pairs": 50,
        "max_train_steps": 200,
        "lora_rank": 16,
    },
    "A100": {  # OPTIMIZED for best quality
        "num_frames": 16,
        "resolution": 512,
        "num_pairs": 200,         # More training data
        "max_train_steps": 500,   # Longer training
        "lora_rank": 32,          # More capacity
    }
}

gpu_config = GPU_CONFIGS[GPU_TYPE]
print(f"Using {GPU_TYPE} GPU - OPTIMIZED configuration")

# Multiple diverse prompts for better generalization
TRAINING_PROMPTS = [
    "cinematic shot, smooth motion, high quality, 8k",
    "drone footage flying over mountains, smooth camera, film quality",
    "slow motion video of water flowing, detailed, seamless",
    "timelapse of sunset clouds, smooth transition, vibrant colors",
    "walking through autumn forest, steady cam, golden hour lighting",
    "ocean waves crashing on beach, smooth motion, aerial view",
    "city street at night, smooth camera movement, neon lights",
    "wildlife documentary shot, smooth tracking, nature footage",
]

# Check existing data
data_dir = "./data/latents"
existing_data = os.path.exists(data_dir) and len([f for f in os.listdir(data_dir) if f.endswith('.pt')]) > 0

if existing_data:
    sample_file = sorted([f for f in os.listdir(data_dir) if f.endswith('.pt')])[0]
    sample = torch.load(os.path.join(data_dir, sample_file))
    latent_shape = sample['latents_w'].shape
    detected_frames = latent_shape[0]
    detected_resolution = latent_shape[2] * 8
    num_pairs_existing = len([f for f in os.listdir(data_dir) if f.endswith('.pt')])
    
    print(f"\nDETECTED existing data: {num_pairs_existing} pairs")
    print(f"To regenerate with new settings, delete ./data/latents/ and re-run")
    
    num_frames = detected_frames
    resolution = detected_resolution
    num_pairs = num_pairs_existing
else:
    print(f"\nWill generate {gpu_config['num_pairs']} pairs with diverse prompts")
    num_frames = gpu_config["num_frames"]
    resolution = gpu_config["resolution"]
    num_pairs = gpu_config["num_pairs"]

CONFIG = {
    "experiment_name": "video_dpo_optimized",
    "output_dir": "./checkpoints",
    "log_dir": "./logs",
    
    "data": {
        "root_dir": "./data/latents",
        "num_pairs": num_pairs,
        "num_frames": num_frames,
        "resolution": resolution,
        "prompts": TRAINING_PROMPTS,  # Multiple prompts
        "jitter_strength": 0.15,      # Stronger jitter
    },
    
    "model": {
        "base_model": "emilianJR/epiCRealism",
        "motion_adapter": "guoyww/animatediff-motion-adapter-v1-5-2",
        "lora_rank": gpu_config["lora_rank"],
        "lora_alpha": gpu_config["lora_rank"] * 2,
        "target_modules": ["to_q", "to_k", "to_v", "to_out.0"],
        "target_modules_pattern": ".*motion_modules.*"
    },
    
    "training": {
        "seed": 42,
        "batch_size": 1,
        "gradient_accumulation_steps": 4,  # Larger effective batch
        "learning_rate": 5e-6,             # Slightly lower for stability
        "max_train_steps": gpu_config["max_train_steps"],
        "beta": 3000,                      # Slightly higher preference strength
        "mixed_precision": "fp16",
        "save_steps": 100,
        "logging_steps": 10,
        "max_grad_norm": 1.0
    }
}

print(f"\n{'='*50}")
print("OPTIMIZED CONFIGURATION")
print(f"{'='*50}")
print(f"  Data pairs: {CONFIG['data']['num_pairs']}")
print(f"  Frames: {CONFIG['data']['num_frames']}")
print(f"  Resolution: {CONFIG['data']['resolution']}x{CONFIG['data']['resolution']}")
print(f"  Training steps: {CONFIG['training']['max_train_steps']}")
print(f"  LoRA rank: {CONFIG['model']['lora_rank']}")
print(f"  Beta: {CONFIG['training']['beta']}")
print(f"  Learning rate: {CONFIG['training']['learning_rate']}")
print(f"  Jitter strength: {CONFIG['data']['jitter_strength']}")
print(f"{'='*50}")

In [18]:
import os
data_dir = "./data/latents"
existing_data = os.path.exists(data_dir) and len([f for f in os.listdir(data_dir) if f.endswith('.pt')]) > 0

if existing_data:
    # Load first file to check dimensions
    sample_file = sorted([f for f in os.listdir(data_dir) if f.endswith('.pt')])[0]
    sample = torch.load(os.path.join(data_dir, sample_file))
    latent_shape = sample['latents_w'].shape
    # Latent shape is [F, C, H, W] where H = resolution/8
    detected_frames = latent_shape[0]
    detected_resolution = latent_shape[2] * 8  # H * 8 = original resolution
    print(f"DETECTED existing data: {detected_frames} frames, {detected_resolution}x{detected_resolution} resolution")
    print(f"Using detected settings to match existing data.\n")
    num_frames = detected_frames
    resolution = detected_resolution
    num_pairs = len([f for f in os.listdir(data_dir) if f.endswith('.pt')])
else:
    # No existing data - use low memory settings for generation
    print("No existing data found. Will generate new data.")
    print("Using LOW MEMORY settings for T4/16GB GPUs.\n")
    num_frames = 8
    resolution = 256
    num_pairs = 10

# Training settings - these can be adjusted regardless of data
max_train_steps = 50
lora_rank = 4  # Keep small for memory

CONFIG = {
    # Experiment settings
    "experiment_name": "video_dpo_colab",
    "output_dir": "./checkpoints",
    "log_dir": "./logs",

    # Data settings
    "data": {
        "root_dir": "./data/latents",
        "num_pairs": num_pairs,
        "num_frames": num_frames,
        "resolution": resolution,
        "prompt": "cinematic shot, high quality, 8k, smooth motion"
    },

    # Model settings
    "model": {
        "base_model": "emilianJR/epiCRealism",
        "motion_adapter": "guoyww/animatediff-motion-adapter-v1-5-2",
        "lora_rank": lora_rank,
        "lora_alpha": lora_rank * 2,
        "target_modules": ["to_q", "to_k", "to_v", "to_out.0"],
        "target_modules_pattern": ".*motion_modules.*"
    },

    # Training settings
    "training": {
        "seed": 42,
        "batch_size": 1,
        "gradient_accumulation_steps": 2,
        "learning_rate": 1e-5,
        "max_train_steps": max_train_steps,
        "beta": 2500,
        "mixed_precision": "fp16",
        "save_steps": 25,
        "logging_steps": 5,
        "max_grad_norm": 1.0
    }
}

print(f"Configuration loaded:")
print(f"  - Data pairs: {CONFIG['data']['num_pairs']}")
print(f"  - Frames: {CONFIG['data']['num_frames']}")
print(f"  - Resolution: {CONFIG['data']['resolution']}x{CONFIG['data']['resolution']}")
print(f"  - Training steps: {CONFIG['training']['max_train_steps']}")
print(f"  - LoRA rank: {CONFIG['model']['lora_rank']}")


DETECTED existing data: 16 frames, 512x512 resolution
Using detected settings to match existing data.

Configuration loaded:
  - Data pairs: 20
  - Frames: 16
  - Resolution: 512x512
  - Training steps: 50
  - LoRA rank: 4


In [19]:
# Save configuration to YAML file
import yaml

os.makedirs("configs", exist_ok=True)
config_path = "configs/colab_config.yaml"

with open(config_path, "w") as f:
    yaml.dump(CONFIG, f, default_flow_style=False)

print(f"Configuration saved to {config_path}")

Configuration saved to configs/colab_config.yaml


In [20]:
# Create necessary directories
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["log_dir"], exist_ok=True)
os.makedirs(CONFIG["data"]["root_dir"], exist_ok=True)

print("Directories created:")
!ls -la

Directories created:
total 24
drwxr-xr-x 6 root root 4096 Dec 11 04:23 .
drwxr-xr-x 1 root root 4096 Dec 11 04:22 ..
drwxr-xr-x 2 root root 4096 Dec 11 04:23 checkpoints
drwxr-xr-x 2 root root 4096 Dec 11 04:23 configs
drwxr-xr-x 3 root root 4096 Dec 11 04:23 data
drwxr-xr-x 2 root root 4096 Dec 11 04:23 logs


---
## 3. Utility Functions

In [21]:
import random
import numpy as np
import torch

def seed_everything(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    """Get the best available device."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Using device: {DEVICE}")
seed_everything(CONFIG["training"]["seed"])

Using device: cuda


---
## 4. Data Generation

Generate preference pairs using the "Jitter Loop" technique:
- **Winner videos**: Temporally consistent (seed-locked generation)
- **Loser videos**: Temporally jittery (per-frame random noise injection)

In [ ]:
def generate_preference_pair(pipe, pair_idx, prompt, num_frames=16, resolution=512, jitter_strength=0.15):
    """
    Generate a winner/loser preference pair with configurable jitter.
    
    Winner: Temporally consistent video (same seed throughout)
    Loser: Temporally jittery video (per-frame noise variation)
    """
    base_seed = CONFIG["training"]["seed"] + pair_idx
    
    # Generate winner (temporally consistent)
    generator = torch.Generator(device=DEVICE).manual_seed(base_seed)
    
    with torch.no_grad():
        output_winner = pipe(
            prompt=prompt,
            num_frames=num_frames,
            height=resolution,
            width=resolution,
            num_inference_steps=25,
            guidance_scale=7.5,
            generator=generator,
            output_type="pt"
        )
        frames_winner = output_winner.frames[0]
    
    # Generate loser with different seed
    generator_loser = torch.Generator(device=DEVICE).manual_seed(base_seed + 10000)
    
    with torch.no_grad():
        output_loser = pipe(
            prompt=prompt,
            num_frames=num_frames,
            height=resolution,
            width=resolution,
            num_inference_steps=25,
            guidance_scale=7.5,
            generator=generator_loser,
            output_type="pt"
        )
        frames_loser = output_loser.frames[0]
    
    # Add temporal jitter to loser - stronger jitter = clearer preference signal
    for i in range(num_frames):
        # Per-frame random noise with configurable strength
        frame_noise = torch.randn_like(frames_loser[i]) * jitter_strength
        # Also add slight color shift per frame for more obvious temporal inconsistency
        color_shift = (torch.rand(3, 1, 1, device=frames_loser.device) - 0.5) * jitter_strength * 0.5
        frames_loser[i] = torch.clamp(frames_loser[i] + frame_noise + color_shift, 0, 1)
    
    # Encode to latent space
    with torch.no_grad():
        latents_winner = pipe.vae.encode(
            frames_winner.to(pipe.vae.dtype)
        ).latent_dist.sample() * pipe.vae.config.scaling_factor
        
        latents_loser = pipe.vae.encode(
            frames_loser.to(pipe.vae.dtype)
        ).latent_dist.sample() * pipe.vae.config.scaling_factor
    
    # Get prompt embeddings
    with torch.no_grad():
        prompt_embeds = pipe.encode_prompt(
            prompt,
            device=DEVICE,
            num_images_per_prompt=1,
            do_classifier_free_guidance=False
        )[0]
    
    return {
        "latents_w": latents_winner.cpu(),
        "latents_l": latents_loser.cpu(),
        "prompt_embeds": prompt_embeds.cpu(),
        "frames_winner": frames_winner.cpu(),
        "frames_loser": frames_loser.cpu()
    }

print("Preference pair generator ready (with configurable jitter)")

In [ ]:
# Generate training data with diverse prompts
print("Loading AnimateDiff pipeline...")
pipe = load_animatediff_pipeline(DEVICE)

num_pairs = CONFIG["data"]["num_pairs"]
num_frames = CONFIG["data"]["num_frames"]
resolution = CONFIG["data"]["resolution"]
data_dir = CONFIG["data"]["root_dir"]
jitter_strength = CONFIG["data"].get("jitter_strength", 0.15)

# Get prompts - either multiple or single
prompts = CONFIG["data"].get("prompts", [CONFIG["data"].get("prompt", "cinematic shot, high quality")])
if isinstance(prompts, str):
    prompts = [prompts]

print(f"\nGenerating {num_pairs} preference pairs...")
print(f"Using {len(prompts)} diverse prompts")
print(f"Frames: {num_frames}, Resolution: {resolution}x{resolution}")
print(f"Jitter strength: {jitter_strength}")

for i in tqdm(range(num_pairs), desc="Generating pairs"):
    try:
        # Cycle through prompts for diversity
        prompt = prompts[i % len(prompts)]
        
        pair_data = generate_preference_pair(
            pipe, i, prompt, num_frames, resolution, jitter_strength
        )
        
        # Save latents
        save_data = {
            "latents_w": pair_data["latents_w"],
            "latents_l": pair_data["latents_l"],
            "prompt_embeds": pair_data["prompt_embeds"]
        }
        
        save_path = os.path.join(data_dir, f"pair_{i:05d}.pt")
        torch.save(save_data, save_path)
        
        # Save first pair as example GIFs
        if i == 0:
            from PIL import Image
            
            winner_frames = [(f.permute(1, 2, 0).numpy() * 255).astype(np.uint8) 
                           for f in pair_data["frames_winner"]]
            loser_frames = [(f.permute(1, 2, 0).numpy() * 255).astype(np.uint8) 
                          for f in pair_data["frames_loser"]]
            
            winner_pil = [Image.fromarray(f) for f in winner_frames]
            loser_pil = [Image.fromarray(f) for f in loser_frames]
            
            winner_pil[0].save(
                "example_winner.gif",
                save_all=True,
                append_images=winner_pil[1:],
                duration=100,
                loop=0
            )
            loser_pil[0].save(
                "example_loser.gif",
                save_all=True,
                append_images=loser_pil[1:],
                duration=100,
                loop=0
            )
            print(f"\nSaved example GIFs")
            print(f"Prompt: {prompt}")
        
        # Clear memory periodically
        if i % 5 == 0:
            torch.cuda.empty_cache()
            gc.collect()
            
    except Exception as e:
        print(f"Error generating pair {i}: {e}")
        continue

print(f"\nData generation complete! Generated {len(os.listdir(data_dir))} pairs.")

# Clean up
del pipe
torch.cuda.empty_cache()
gc.collect()

In [24]:
# Generate training data
print("Loading AnimateDiff pipeline...")
pipe = load_animatediff_pipeline(DEVICE)

num_pairs = CONFIG["data"]["num_pairs"]
prompt = CONFIG["data"]["prompt"]
num_frames = CONFIG["data"]["num_frames"]
resolution = CONFIG["data"]["resolution"]
data_dir = CONFIG["data"]["root_dir"]

print(f"\nGenerating {num_pairs} preference pairs...")
print(f"Prompt: {prompt}")
print(f"Frames: {num_frames}, Resolution: {resolution}x{resolution}")

# for i in tqdm(range(num_pairs), desc="Generating pairs"):
#     try:
#         pair_data = generate_preference_pair(
#             pipe, i, prompt, num_frames, resolution
#         )

#         # Save latents (remove frames to save space)
#         save_data = {
#             "latents_w": pair_data["latents_w"],
#             "latents_l": pair_data["latents_l"],
#             "prompt_embeds": pair_data["prompt_embeds"]
#         }

#         save_path = os.path.join(data_dir, f"pair_{i:05d}.pt")
#         torch.save(save_data, save_path)

#         # Save first pair as example GIFs
#         if i == 0:
#             from PIL import Image

#             # Convert to PIL images
#             winner_frames = [(f.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
#                            for f in pair_data["frames_winner"]]
#             loser_frames = [(f.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
#                           for f in pair_data["frames_loser"]]

#             winner_pil = [Image.fromarray(f) for f in winner_frames]
#             loser_pil = [Image.fromarray(f) for f in loser_frames]

#             winner_pil[0].save(
#                 "example_winner.gif",
#                 save_all=True,
#                 append_images=winner_pil[1:],
#                 duration=100,
#                 loop=0
#             )
#             loser_pil[0].save(
#                 "example_loser.gif",
#                 save_all=True,
#                 append_images=loser_pil[1:],
#                 duration=100,
#                 loop=0
#             )
#             print(f"\nSaved example GIFs: example_winner.gif, example_loser.gif")

#         # Clear memory periodically
#         if i % 5 == 0:
#             torch.cuda.empty_cache()
#             gc.collect()

#     except Exception as e:
#         print(f"Error generating pair {i}: {e}")
#         continue

print(f"\nData generation complete! Generated {len(os.listdir(data_dir))} pairs.")

# Clean up pipeline
del pipe
torch.cuda.empty_cache()
gc.collect()

Loading AnimateDiff pipeline...
Loading motion adapter...


The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading AnimateDiff pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]


Generating 20 preference pairs...
Prompt: cinematic shot, high quality, 8k, smooth motion
Frames: 16, Resolution: 512x512

Data generation complete! Generated 20 pairs.


164

In [11]:
# Display example GIFs
from IPython.display import Image, display, HTML

print("Winner video (temporally consistent):")
display(Image(filename="example_winner.gif"))

print("\nLoser video (temporally jittery):")
display(Image(filename="example_loser.gif"))

Winner video (temporally consistent):


FileNotFoundError: [Errno 2] No such file or directory: 'example_winner.gif'

In [25]:
# Validate generated data
print("Validating generated data...\n")

data_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pt")])
print(f"Found {len(data_files)} data files")

# Check first file
sample = torch.load(os.path.join(data_dir, data_files[0]))
print(f"\nSample data structure:")
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
        print(f"         min={value.min():.4f}, max={value.max():.4f}, mean={value.mean():.4f}")

# Validate all files
valid_count = 0
for f in tqdm(data_files, desc="Validating"):
    try:
        data = torch.load(os.path.join(data_dir, f))
        assert "latents_w" in data and "latents_l" in data and "prompt_embeds" in data
        assert not torch.isnan(data["latents_w"]).any()
        assert not torch.isnan(data["latents_l"]).any()
        valid_count += 1
    except Exception as e:
        print(f"Invalid file {f}: {e}")

print(f"\nValidation complete: {valid_count}/{len(data_files)} files valid")

Validating generated data...

Found 20 data files

Sample data structure:
  latents_w: shape=torch.Size([16, 4, 64, 64]), dtype=torch.float16
         min=-16.0938, max=12.0703, mean=0.1461
  latents_l: shape=torch.Size([16, 4, 64, 64]), dtype=torch.float16
         min=-10.4375, max=9.6016, mean=-0.0165
  prompt_embeds: shape=torch.Size([1, 77, 768]), dtype=torch.float16
         min=-28.1250, max=33.0000, mean=-0.1096


Validating:   0%|          | 0/20 [00:00<?, ?it/s]


Validation complete: 20/20 files valid


---
## 5. DPO Training

Train LoRA weights on motion modules using Direct Preference Optimization.

In [26]:
from torch.utils.data import Dataset, DataLoader
import re

class VideoDPODataset(Dataset):
    """Dataset for DPO training with winner/loser preference pairs."""

    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pt")])
        print(f"Loaded dataset with {len(self.files)} pairs")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(os.path.join(self.data_dir, self.files[idx]))
        return {
            "latents_w": data["latents_w"],
            "latents_l": data["latents_l"],
            "prompt_embeds": data["prompt_embeds"]
        }

# Create dataset and dataloader
dataset = VideoDPODataset(CONFIG["data"]["root_dir"])
dataloader = DataLoader(
    dataset,
    batch_size=CONFIG["training"]["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print(f"DataLoader ready with {len(dataloader)} batches")

Loaded dataset with 20 pairs
DataLoader ready with 20 batches


In [27]:
from peft import LoraConfig, get_peft_model
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
import copy
import gc

def create_dpo_models(device):
    """
    Create policy (trainable) and reference (frozen) models for DPO.
    Memory-optimized version for Colab.
    """
    print("Loading base model and motion adapter...")

    # Clear any existing memory
    torch.cuda.empty_cache()
    gc.collect()

    # Load motion adapter - use fp16 to save memory
    motion_adapter = MotionAdapter.from_pretrained(
        CONFIG["model"]["motion_adapter"],
        torch_dtype=torch.float16
    )

    # Load pipeline in fp16
    pipe = AnimateDiffPipeline.from_pretrained(
        CONFIG["model"]["base_model"],
        motion_adapter=motion_adapter,
        torch_dtype=torch.float16
    )

    # Move to device
    pipe = pipe.to(device)

    # Get UNet for training
    unet = pipe.unet

    # Configure LoRA for motion modules only
    target_modules = []
    pattern = re.compile(CONFIG["model"]["target_modules_pattern"])

    for name, module in unet.named_modules():
        if pattern.match(name):
            for target in CONFIG["model"]["target_modules"]:
                if target in name:
                    target_modules.append(name)
                    break

    target_modules = list(set(target_modules))

    if not target_modules:
        target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
        print(f"Using default target modules: {target_modules}")
    else:
        print(f"Found {len(target_modules)} motion module layers to target")

    # Create LoRA config
    lora_config = LoraConfig(
        r=CONFIG["model"]["lora_rank"],
        lora_alpha=CONFIG["model"]["lora_alpha"],
        target_modules=["to_q", "to_k", "to_v", "to_out.0"],
        lora_dropout=0.0,
        bias="none"
    )

    # MEMORY OPTIMIZATION: Instead of deep copying, we'll use a single model
    # and disable/enable LoRA adapters for reference vs policy computations
    print("Applying LoRA to UNet (memory-optimized, no deep copy)...")

    # Apply LoRA - this modifies unet in place
    policy_unet = get_peft_model(unet, lora_config)
    policy_unet.print_trainable_parameters()

    # Enable gradient checkpointing to save memory during backward pass
    if hasattr(policy_unet, 'enable_gradient_checkpointing'):
        policy_unet.enable_gradient_checkpointing()
        print("Gradient checkpointing enabled")

    # We won't create a separate reference model
    # Instead, we'll use disable_adapter() during forward pass
    ref_unet = None  # Will use policy_unet with adapters disabled

    torch.cuda.empty_cache()
    gc.collect()

    print(f"\nMemory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

    return policy_unet, ref_unet, pipe

print("Model creation function ready (memory-optimized)")

Model creation function ready (memory-optimized)


In [31]:
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

def compute_dpo_loss(policy_unet, ref_unet, batch, timesteps, noise, beta=2500):
    """
    Compute DPO loss for video preference learning.

    Memory-optimized: Uses single model with LoRA adapter toggling.
    Fixed for AnimateDiff UNet - prompt embeddings must be repeated per frame.
    """
    latents_w = batch["latents_w"].to(DEVICE)
    latents_l = batch["latents_l"].to(DEVICE)
    prompt_embeds = batch["prompt_embeds"].to(DEVICE)

    # Fix prompt_embeds shape: need [B, seq_len, hidden]
    while prompt_embeds.dim() > 3:
        prompt_embeds = prompt_embeds.squeeze(1)
    if prompt_embeds.dim() == 2:
        prompt_embeds = prompt_embeds.unsqueeze(0)

    # Get dimensions from data
    # Data shape from DataLoader: [B, F, C, H, W]
    if latents_w.dim() == 4:
        # Single sample: [F, C, H, W] -> [1, F, C, H, W]
        latents_w = latents_w.unsqueeze(0)
        latents_l = latents_l.unsqueeze(0)

    batch_size, num_frames, channels, height, width = latents_w.shape

    # AnimateDiff UNet expects [B, C, F, H, W]
    latents_w = latents_w.permute(0, 2, 1, 3, 4).contiguous()
    latents_l = latents_l.permute(0, 2, 1, 3, 4).contiguous()

    # Generate noise matching latent shape [B, C, F, H, W]
    if noise is None:
        noise = torch.randn_like(latents_w)
    elif noise.shape != latents_w.shape:
        noise = torch.randn_like(latents_w)

    # Ensure timesteps has correct batch size
    if timesteps.shape[0] != batch_size:
        timesteps = torch.randint(0, 1000, (batch_size,), device=DEVICE)

    # Forward diffusion (add noise)
    alpha_t = (1 - timesteps.float() / 1000).view(-1, 1, 1, 1, 1)
    sigma_t = (timesteps.float() / 1000).view(-1, 1, 1, 1, 1)

    noisy_w = alpha_t.sqrt() * latents_w + sigma_t.sqrt() * noise
    noisy_l = alpha_t.sqrt() * latents_l + sigma_t.sqrt() * noise

    # CRITICAL FIX: AnimateDiff requires prompt_embeds to be repeated for each frame
    # Shape: [B, seq_len, hidden] -> [B * num_frames, seq_len, hidden]
    if prompt_embeds.shape[0] != batch_size:
        prompt_embeds = prompt_embeds.expand(batch_size, -1, -1)

    # Repeat for each frame (this is what AnimateDiff pipeline does internally)
    prompt_embeds_video = prompt_embeds.repeat_interleave(num_frames, dim=0)

    # --- REFERENCE MODEL PREDICTIONS (adapters disabled) ---
    with torch.no_grad():
        policy_unet.disable_adapter_layers()

        ref_pred_w = policy_unet(
            noisy_w,
            timesteps,
            encoder_hidden_states=prompt_embeds_video,
            return_dict=False
        )[0]
        ref_pred_l = policy_unet(
            noisy_l,
            timesteps,
            encoder_hidden_states=prompt_embeds_video,
            return_dict=False
        )[0]

        policy_unet.enable_adapter_layers()

    # --- POLICY MODEL PREDICTIONS (adapters enabled) ---
    policy_pred_w = policy_unet(
        noisy_w,
        timesteps,
        encoder_hidden_states=prompt_embeds_video,
        return_dict=False
    )[0]
    policy_pred_l = policy_unet(
        noisy_l,
        timesteps,
        encoder_hidden_states=prompt_embeds_video,
        return_dict=False
    )[0]

    # Compute MSE errors
    policy_error_w = F.mse_loss(policy_pred_w, noise, reduction="none").mean(dim=[1,2,3,4])
    policy_error_l = F.mse_loss(policy_pred_l, noise, reduction="none").mean(dim=[1,2,3,4])
    ref_error_w = F.mse_loss(ref_pred_w, noise, reduction="none").mean(dim=[1,2,3,4])
    ref_error_l = F.mse_loss(ref_pred_l, noise, reduction="none").mean(dim=[1,2,3,4])

    # Implicit rewards
    reward_w = ref_error_w - policy_error_w
    reward_l = ref_error_l - policy_error_l

    # DPO loss
    reward_diff = reward_w - reward_l
    loss = -F.logsigmoid(beta * reward_diff).mean()

    return loss, reward_diff.mean().item()

print("DPO loss function ready (with AnimateDiff frame handling)")

DPO loss function ready (with AnimateDiff frame handling)


In [32]:
# Initialize models (memory-optimized)
print("Initializing DPO models...\n")

# Clear memory before loading
torch.cuda.empty_cache()
gc.collect()

policy_unet, ref_unet, pipe = create_dpo_models(DEVICE)

# Setup optimizer - only optimize LoRA parameters
optimizer = AdamW(
    filter(lambda p: p.requires_grad, policy_unet.parameters()),
    lr=CONFIG["training"]["learning_rate"],
    betas=(0.9, 0.999),
    weight_decay=0.01
)

# Setup mixed precision scaler
scaler = GradScaler('cuda') if CONFIG["training"]["mixed_precision"] == "fp16" else None

print(f"\nOptimizer: AdamW with lr={CONFIG['training']['learning_rate']}")
print(f"Mixed precision: {CONFIG['training']['mixed_precision']}")
print(f"Reference model: Using adapter toggling (memory-efficient)")

# Print memory status
print(f"\nGPU Memory after model init:")
print(f"  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

Initializing DPO models...

Loading base model and motion adapter...


The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Found 168 motion module layers to target
Applying LoRA to UNet (memory-optimized, no deep copy)...
trainable params: 2,005,504 || all params: 1,314,735,748 || trainable%: 0.1525
Gradient checkpointing enabled

Memory allocated: 8.32 GB
Memory reserved: 8.42 GB

Optimizer: AdamW with lr=1e-05
Mixed precision: fp16
Reference model: Using adapter toggling (memory-efficient)

GPU Memory after model init:
  Allocated: 6.98 GB
  Reserved: 8.42 GB


In [33]:
# Training loop
print("Starting DPO training...\n")

max_steps = CONFIG["training"]["max_train_steps"]
grad_accum_steps = CONFIG["training"]["gradient_accumulation_steps"]
beta = CONFIG["training"]["beta"]
save_steps = CONFIG["training"]["save_steps"]
logging_steps = CONFIG["training"]["logging_steps"]
max_grad_norm = CONFIG["training"]["max_grad_norm"]

policy_unet.train()
global_step = 0
total_loss = 0
total_reward_diff = 0

progress_bar = tqdm(total=max_steps, desc="Training")
training_history = {"loss": [], "reward_diff": [], "step": []}

while global_step < max_steps:
    for batch in dataloader:
        if global_step >= max_steps:
            break

        # Sample timesteps (noise is generated inside compute_dpo_loss)
        batch_size = batch["latents_w"].shape[0]
        timesteps = torch.randint(0, 1000, (batch_size,), device=DEVICE)
        noise = None  # Will be generated inside compute_dpo_loss

        # Forward pass with mixed precision
        if scaler is not None:
            with autocast('cuda'):
                loss, reward_diff = compute_dpo_loss(
                    policy_unet, ref_unet, batch, timesteps, noise, beta
                )
                loss = loss / grad_accum_steps

            scaler.scale(loss).backward()
        else:
            loss, reward_diff = compute_dpo_loss(
                policy_unet, ref_unet, batch, timesteps, noise, beta
            )
            loss = loss / grad_accum_steps
            loss.backward()

        total_loss += loss.item() * grad_accum_steps
        total_reward_diff += reward_diff

        # Gradient accumulation step
        if (global_step + 1) % grad_accum_steps == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(policy_unet.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(policy_unet.parameters(), max_grad_norm)
                optimizer.step()

            optimizer.zero_grad()

        global_step += 1
        progress_bar.update(1)

        # Logging
        if global_step % logging_steps == 0:
            avg_loss = total_loss / logging_steps
            avg_reward_diff = total_reward_diff / logging_steps

            training_history["loss"].append(avg_loss)
            training_history["reward_diff"].append(avg_reward_diff)
            training_history["step"].append(global_step)

            progress_bar.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "reward_diff": f"{avg_reward_diff:.4f}"
            })

            total_loss = 0
            total_reward_diff = 0

        # Save checkpoint
        if global_step % save_steps == 0:
            checkpoint_dir = os.path.join(CONFIG["output_dir"], f"checkpoint-{global_step}")
            os.makedirs(checkpoint_dir, exist_ok=True)
            policy_unet.save_pretrained(checkpoint_dir)
            print(f"\nSaved checkpoint to {checkpoint_dir}")

progress_bar.close()

# Save final checkpoint
final_checkpoint = os.path.join(CONFIG["output_dir"], "final")
os.makedirs(final_checkpoint, exist_ok=True)
policy_unet.save_pretrained(final_checkpoint)
print(f"\nTraining complete! Final checkpoint saved to {final_checkpoint}")

Starting DPO training...



Training:   0%|          | 0/50 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 120.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 54.12 MiB is free. Process 250226 has 14.69 GiB memory in use. Of the allocated memory 14.27 GiB is allocated by PyTorch, and 297.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(training_history["step"], training_history["loss"])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("DPO Training Loss")
axes[0].grid(True)

axes[1].plot(training_history["step"], training_history["reward_diff"])
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Reward Difference")
axes[1].set_title("Winner-Loser Reward Difference")
axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

print("Training curves saved to training_curves.png")

---
## 6. Inference & Comparison

Generate side-by-side comparison videos: base model vs DPO-aligned model.

In [ ]:
# Generate comparison videos with comprehensive test prompts
test_prompts = [
    # ===== NATURE & LANDSCAPES =====
    "cinematic shot of a sunset over mountains, high quality, smooth motion",
    "ocean waves crashing on rocky shore, slow motion, 8k quality",
    "aerial drone footage of a winding river through forest, smooth camera",
    "northern lights dancing in night sky, timelapse, vibrant colors",
    "waterfall cascading down mossy rocks, slow motion, mist rising",
    "desert sand dunes with wind blowing sand, golden hour lighting",
    "snow falling gently in a winter forest, peaceful, soft lighting",
    "lightning storm over open plains, dramatic, high contrast",
    
    # ===== ANIMALS & WILDLIFE =====
    "a cat walking gracefully, high quality video, 8k",
    "a dog running through a field of flowers, slow motion, golden hour",
    "birds flying in formation across sunset sky, smooth tracking shot",
    "fish swimming in clear blue ocean, underwater footage, smooth motion",
    "butterfly landing on a flower, macro shot, gentle movement",
    "horse galloping across open meadow, slow motion, dust particles",
    "jellyfish floating in dark water, bioluminescent, ethereal",
    
    # ===== URBAN & ARCHITECTURE =====
    "busy city street at night with neon lights, smooth camera movement",
    "timelapse of clouds moving over city skyline, seamless transitions",
    "car driving through rainy city streets, reflections, cinematic",
    "elevator ascending glass building, city view expanding, smooth",
    "train passing through station, motion blur, dynamic shot",
    "fireworks exploding over cityscape, celebration, vibrant colors",
    
    # ===== ABSTRACT & ARTISTIC =====
    "colorful smoke swirling in slow motion, abstract, high detail",
    "water droplets falling into calm pool, macro shot, smooth motion",
    "ink dropping into water, swirling patterns, mesmerizing",
    "abstract liquid metal morphing shapes, reflective, sci-fi",
    "kaleidoscope patterns shifting and rotating, hypnotic, colorful",
    "paint dripping down canvas, abstract expressionism, vibrant",
    
    # ===== PEOPLE & ACTION =====
    "dancer performing ballet in spotlight, slow motion, cinematic",
    "martial artist performing kata, fluid movements, dramatic lighting",
    "hands playing piano keys, close-up, emotional, soft lighting",
    "chef cooking with flames in kitchen, dynamic, professional",
    "athlete running on track, slow motion, determination, sunrise",
    
    # ===== FANTASY & SCI-FI =====
    "magical portal opening with energy swirls, fantasy, glowing",
    "spaceship flying through asteroid field, sci-fi, dramatic",
    "dragon flying through cloudy sky, fantasy epic, majestic",
    "robot walking in futuristic city, cyberpunk, neon lights",
    "fairy dust particles floating in enchanted forest, magical",
    
    # ===== FOOD & OBJECTS =====
    "coffee being poured into cup, steam rising, cozy atmosphere",
    "candle flame flickering gently, warm lighting, peaceful",
    "clock gears turning mechanism, macro, steampunk aesthetic",
    "champagne bubbles rising in glass, celebration, elegant",
    "leaves falling from autumn tree, gentle breeze, warm colors",
    
    # ===== WEATHER & ELEMENTS =====
    "rain drops hitting window glass, close-up, melancholic mood",
    "fire burning in fireplace, crackling, cozy winter night",
    "clouds forming and dissipating, timelapse, dramatic sky",
    "tornado forming in distance, dramatic weather, ominous",
    "sun rays breaking through storm clouds, hope, dramatic lighting",
]

# Prompt selection options
PROMPT_MODE = "subset"  # Options: "subset", "all", "category"
SELECTED_CATEGORY = "nature"  # Used when PROMPT_MODE = "category"

# Category indices for selective testing
CATEGORIES = {
    "nature": (0, 8),      # indices 0-7
    "animals": (8, 15),    # indices 8-14
    "urban": (15, 21),     # indices 15-20
    "abstract": (21, 27),  # indices 21-26
    "people": (27, 32),    # indices 27-31
    "fantasy": (32, 37),   # indices 32-36
    "objects": (37, 42),   # indices 37-41
    "weather": (42, 47),   # indices 42-46
}

if PROMPT_MODE == "all":
    selected_prompts = test_prompts
    print(f"Running ALL {len(selected_prompts)} test prompts")
    
elif PROMPT_MODE == "category":
    start, end = CATEGORIES.get(SELECTED_CATEGORY, (0, 5))
    selected_prompts = test_prompts[start:end]
    print(f"Running {SELECTED_CATEGORY.upper()} category: {len(selected_prompts)} prompts")
    
else:  # subset - one from each category
    selected_prompts = [
        test_prompts[0],   # nature: sunset mountains
        test_prompts[4],   # nature: waterfall
        test_prompts[8],   # animals: cat
        test_prompts[13],  # animals: horse
        test_prompts[15],  # urban: city night
        test_prompts[18],  # urban: rainy streets
        test_prompts[21],  # abstract: smoke
        test_prompts[24],  # abstract: ink in water
        test_prompts[27],  # people: ballet dancer
        test_prompts[32],  # fantasy: magic portal
        test_prompts[37],  # objects: coffee
        test_prompts[42],  # weather: rain on window
    ]
    print(f"Running SUBSET: {len(selected_prompts)} diverse prompts (one from each category)")

test_prompts = selected_prompts
checkpoint_path = os.path.join(CONFIG["output_dir"], "final")

print(f"\nUsing checkpoint: {checkpoint_path}")
print(f"\nTest prompts:")
for i, p in enumerate(test_prompts):
    print(f"  {i+1}. {p[:65]}{'...' if len(p) > 65 else ''}")

print(f"\n{'='*60}")
print("Starting video generation...")
print(f"{'='*60}\n")

comparison_results = generate_comparison_videos(
    checkpoint_path,
    test_prompts,
    num_frames=CONFIG["data"]["num_frames"],
    resolution=CONFIG["data"]["resolution"]
)

In [ ]:
# Save and display comparison GIFs
from PIL import Image as PILImage
from IPython.display import Image as IPyImage, display

os.makedirs("comparison_results", exist_ok=True)

for i, result in enumerate(comparison_results):
    prompt = result["prompt"][:50] + "..." if len(result["prompt"]) > 50 else result["prompt"]
    print(f"\n--- Prompt: {prompt} ---")
    
    # Save base GIF
    base_path = f"comparison_results/comparison_{i}_base.gif"
    result["base_frames"][0].save(
        base_path,
        save_all=True,
        append_images=result["base_frames"][1:],
        duration=100,
        loop=0
    )
    
    print("\nBase Model:")
    display(IPyImage(filename=base_path))
    
    # Save DPO GIF if available
    if result["dpo_frames"] is not None:
        dpo_path = f"comparison_results/comparison_{i}_dpo.gif"
        result["dpo_frames"][0].save(
            dpo_path,
            save_all=True,
            append_images=result["dpo_frames"][1:],
            duration=100,
            loop=0
        )
        
        print("\nDPO-Aligned Model:")
        display(IPyImage(filename=dpo_path))

print("\nComparison videos saved to comparison_results/")

In [ ]:
# Save and display comparison GIFs
from PIL import Image

os.makedirs("comparison_results", exist_ok=True)

for i, result in enumerate(comparison_results):
    prompt = result["prompt"][:50] + "..." if len(result["prompt"]) > 50 else result["prompt"]
    print(f"\n--- Prompt: {prompt} ---")

    # Save base GIF
    base_path = f"comparison_results/comparison_{i}_base.gif"
    result["base_frames"][0].save(
        base_path,
        save_all=True,
        append_images=result["base_frames"][1:],
        duration=100,
        loop=0
    )

    print("\nBase Model:")
    display(Image(filename=base_path))

    # Save DPO GIF if available
    if result["dpo_frames"] is not None:
        dpo_path = f"comparison_results/comparison_{i}_dpo.gif"
        result["dpo_frames"][0].save(
            dpo_path,
            save_all=True,
            append_images=result["dpo_frames"][1:],
            duration=100,
            loop=0
        )

        print("\nDPO-Aligned Model:")
        display(Image(filename=dpo_path))

print("\nComparison videos saved to comparison_results/")

---
## 7. Quantitative Evaluation

Compute temporal consistency metrics:
- **Warping Error**: Optical flow based consistency (lower is better)
- **Frame Difference**: Average flickering between frames (lower is better)

In [ ]:
import cv2
from scipy import ndimage

def compute_optical_flow(frame1, frame2):
    """Compute dense optical flow between two frames."""
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_RGB2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        gray1, gray2, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )
    return flow

def compute_warping_error(frames):
    """Compute warping error using optical flow."""
    errors = []

    for i in range(len(frames) - 1):
        frame1 = np.array(frames[i])
        frame2 = np.array(frames[i + 1])

        # Compute optical flow
        flow = compute_optical_flow(frame1, frame2)

        # Warp frame1 to predict frame2
        h, w = frame1.shape[:2]
        flow_map = np.column_stack((
            (np.arange(w) + flow[..., 0]).flatten(),
            (np.arange(h)[:, None] + flow[..., 1]).flatten()
        )).reshape(h, w, 2)

        # Simple warping using flow
        warped = np.zeros_like(frame1)
        for c in range(3):
            warped[..., c] = ndimage.map_coordinates(
                frame1[..., c],
                [flow_map[..., 1], flow_map[..., 0]],
                order=1, mode='nearest'
            )

        # Compute MSE between warped and actual
        error = np.mean((warped.astype(float) - frame2.astype(float)) ** 2)
        errors.append(error)

    return np.mean(errors)

def compute_frame_difference(frames):
    """Compute average absolute difference between consecutive frames."""
    diffs = []

    for i in range(len(frames) - 1):
        frame1 = np.array(frames[i]).astype(float)
        frame2 = np.array(frames[i + 1]).astype(float)
        diff = np.mean(np.abs(frame1 - frame2))
        diffs.append(diff)

    return np.mean(diffs)

def evaluate_temporal_consistency(frames):
    """Evaluate temporal consistency of video frames."""
    return {
        "warping_error": compute_warping_error(frames),
        "frame_difference": compute_frame_difference(frames)
    }

print("Evaluation functions ready")

In [ ]:
# Evaluate comparison results
print("Evaluating temporal consistency...\n")

evaluation_results = []

for i, result in enumerate(comparison_results):
    prompt = result["prompt"][:50] + "..."
    print(f"Evaluating: {prompt}")

    # Evaluate base model
    base_metrics = evaluate_temporal_consistency(result["base_frames"])
    print(f"  Base Model:")
    print(f"    Warping Error: {base_metrics['warping_error']:.2f}")
    print(f"    Frame Diff: {base_metrics['frame_difference']:.2f}")

    # Evaluate DPO model
    if result["dpo_frames"] is not None:
        dpo_metrics = evaluate_temporal_consistency(result["dpo_frames"])
        print(f"  DPO Model:")
        print(f"    Warping Error: {dpo_metrics['warping_error']:.2f}")
        print(f"    Frame Diff: {dpo_metrics['frame_difference']:.2f}")

        # Improvement
        warp_improvement = (base_metrics['warping_error'] - dpo_metrics['warping_error']) / base_metrics['warping_error'] * 100
        diff_improvement = (base_metrics['frame_difference'] - dpo_metrics['frame_difference']) / base_metrics['frame_difference'] * 100
        print(f"  Improvement: Warping {warp_improvement:+.1f}%, Frame Diff {diff_improvement:+.1f}%")

        evaluation_results.append({
            "prompt": result["prompt"],
            "base": base_metrics,
            "dpo": dpo_metrics
        })
    else:
        evaluation_results.append({
            "prompt": result["prompt"],
            "base": base_metrics,
            "dpo": None
        })

    print()

# Save evaluation results
torch.save(evaluation_results, "evaluation_results.pt")
print("Evaluation results saved to evaluation_results.pt")

In [ ]:
# Summary statistics
if evaluation_results and evaluation_results[0]["dpo"] is not None:
    base_warp = np.mean([r["base"]["warping_error"] for r in evaluation_results])
    base_diff = np.mean([r["base"]["frame_difference"] for r in evaluation_results])
    dpo_warp = np.mean([r["dpo"]["warping_error"] for r in evaluation_results if r["dpo"]])
    dpo_diff = np.mean([r["dpo"]["frame_difference"] for r in evaluation_results if r["dpo"]])

    print("="*50)
    print("SUMMARY")
    print("="*50)
    print(f"\n{'Metric':<20} {'Base':<15} {'DPO':<15} {'Change':<15}")
    print("-"*60)
    print(f"{'Warping Error':<20} {base_warp:<15.2f} {dpo_warp:<15.2f} {(dpo_warp-base_warp)/base_warp*100:+.1f}%")
    print(f"{'Frame Difference':<20} {base_diff:<15.2f} {dpo_diff:<15.2f} {(dpo_diff-base_diff)/base_diff*100:+.1f}%")
    print("\nNote: Lower values indicate better temporal consistency.")
else:
    print("No DPO results available for comparison.")

---
## 8. Download Results

Download all generated files and checkpoints.

In [ ]:
# Create a zip file with all results
import shutil

# List all output files
print("Generated files:")
!ls -la *.gif *.png *.pt 2>/dev/null || echo "No files in root"
print("\nCheckpoints:")
!ls -la checkpoints/ 2>/dev/null || echo "No checkpoints"
print("\nComparison results:")
!ls -la comparison_results/ 2>/dev/null || echo "No comparison results"

In [ ]:
# Create zip archive for download
import zipfile

zip_path = "video_dpo_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add GIFs
    for f in ["example_winner.gif", "example_loser.gif", "training_curves.png"]:
        if os.path.exists(f):
            zf.write(f)

    # Add comparison results
    if os.path.exists("comparison_results"):
        for f in os.listdir("comparison_results"):
            zf.write(os.path.join("comparison_results", f))

    # Add evaluation results
    if os.path.exists("evaluation_results.pt"):
        zf.write("evaluation_results.pt")

    # Add final checkpoint
    final_ckpt = os.path.join(CONFIG["output_dir"], "final")
    if os.path.exists(final_ckpt):
        for f in os.listdir(final_ckpt):
            zf.write(os.path.join(final_ckpt, f), os.path.join("checkpoint_final", f))

print(f"Created {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

In [ ]:
# Download the zip file (Colab)
try:
    from google.colab import files
    files.download(zip_path)
    print("Download started!")
except ImportError:
    print(f"Not running in Colab. File saved at: {os.path.abspath(zip_path)}")

---
## Summary

This notebook demonstrated the complete Video-DPO pipeline:

1. **Data Generation**: Created preference pairs using the "Jitter Loop" technique
2. **DPO Training**: Optimized LoRA weights on motion modules to prefer temporally consistent videos
3. **Inference**: Generated comparison videos between base and DPO-aligned models
4. **Evaluation**: Computed quantitative metrics for temporal consistency

### Tips for Better Results:
- Increase `num_pairs` to 100-500 for more training data
- Increase `max_train_steps` to 500-1000 for longer training
- Use A100 GPU for faster processing
- Experiment with different `beta` values (1000-5000) for stricter/looser preference learning

### References:
- [Diffusion-DPO Paper](https://arxiv.org/abs/2311.12908)
- [AnimateDiff](https://github.com/guoyww/AnimateDiff)
- [PEFT/LoRA](https://github.com/huggingface/peft)